# Fine-tuning the Base Model on Google Colab

Runs `python -m pipeline.measure_fine_tuning` on a Colab GPU. The script is
the same one that runs on a desktop, where it takes hours on the CPU; see
[the experiment's page](../docs/experiments/fine-tuning.md) for what it
measures and how.

Before running: **Runtime → Change runtime type → T4 GPU** (or any GPU).
Then run the cells in order. The last one downloads the results file, which
belongs in `docs/experiments/results/fine-tuning.json` of the repository.

In [ ]:
# The branch or commit to measure. Colab clones from GitHub, so it must
# have been pushed there.
REF = "master"

!nvidia-smi --query-gpu=name,memory.total --format=csv

## The code and its dependencies

Colab already has a CUDA build of PyTorch, which satisfies the `models`
extra, so pip leaves it in place and adds the rest, CSSLab's maia3 package
among them.

In [ ]:
!git clone --quiet https://github.com/CamilleBochard/ChessMe.git
%cd ChessMe
!git checkout --quiet {REF}
!git log --oneline -1
!pip install --quiet -e '.[models]'

## The dataset and the weights

The raw games are in the repository, so the dataset is rebuilt here exactly
as on the desktop: the split between training games and the Test Set
depends only on each game's id. The weights are downloaded from the pinned
URLs and checked against their recorded hashes.

In [ ]:
!python -m pipeline.build_dataset
!python -m pipeline.fetch_models

## The run

Tables go to the output below and to `fine-tuning-tables.md`; progress
lines show each learning rate and epoch as it finishes. The first lines
compare the PyTorch Baseline with the counts the engine recorded: a warning
there means the comparison cannot be trusted.

In [ ]:
!python -m pipeline.measure_fine_tuning --device cuda | tee fine-tuning-tables.md

In [ ]:
from google.colab import files

files.download("docs/experiments/results/fine-tuning.json")
files.download("fine-tuning-tables.md")